In [ ]:
# with 0.2
#smallpox
# -*- coding: utf-8 -*-
"""
Complete implementation of MCPR (Multiplex Combined PageRank) according to the paper
"An Immunization Method using a Context-based Centrality in Multiplex Networks"

This code includes:
1. Correct implementation of MCPR according to equation (7) of the paper
2. SIR-UA epidemic simulation model
3. Various immunization algorithms
4. Analysis and comparison of results

Modified version to work with CSV datasets
"""

import networkx as nx
import numpy as np
import pandas as pd
from multiprocessing import Pool, cpu_count
import time
import warnings
import matplotlib.pyplot as plt
import os
import sys

# Encoding and warnings settings
warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams['font.family'] = 'DejaVu Sans'

# =============================================================================
# Section 1: Centrality Calculation Functions and Correct MCPR
# =============================================================================

def get_centrality(graph: nx.Graph, measure_code: int) -> dict:
    """
    Calculate various centrality measures

    Args:
        graph: NetworkX graph
        measure_code: centrality type code (1-5)
                     1=degree, 2=betweenness, 3=closeness, 4=eigenvector, 5=pagerank

    Returns:
        dict: centrality values for each node
    """
    if measure_code == 1:
        return nx.degree_centrality(graph)

    elif measure_code == 2:
        return nx.betweenness_centrality(graph)

    elif measure_code == 3:  # closeness - recommended in the paper for virtual layer
        if not nx.is_connected(graph):
            # For disconnected networks
            closeness_vals = {}
            for component in nx.connected_components(graph):
                if len(component) > 1:  # Only components with more than one node
                    subgraph = graph.subgraph(component)
                    component_closeness = nx.closeness_centrality(subgraph)
                    closeness_vals.update(component_closeness)

            # Isolated or disconnected nodes
            for node in graph.nodes():
                if node not in closeness_vals:
                    closeness_vals[node] = 0.0

            return closeness_vals
        else:
            return nx.closeness_centrality(graph)

    elif measure_code == 4:  # eigenvector
        if not nx.is_connected(graph):
            # For disconnected graph, use only the largest connected component
            largest_cc = max(nx.connected_components(graph), key=len)
            subgraph = graph.subgraph(largest_cc)
            try:
                eigen_vals = nx.eigenvector_centrality(subgraph, max_iter=1000)
                full_eigen = {node: 0.0 for node in graph.nodes()}
                full_eigen.update(eigen_vals)
                return full_eigen
            except nx.NetworkXError:
                # If not converged, use degree centrality
                return nx.degree_centrality(graph)
        else:
            try:
                return nx.eigenvector_centrality(graph, max_iter=1000)
            except nx.NetworkXError:
                return nx.degree_centrality(graph)

    elif measure_code == 5:  # pagerank
        return nx.pagerank(graph, alpha=0.85, max_iter=1000)

    else:
        raise ValueError("measure_code must be between 1 and 5")


def mcpr_correct(g_virtual: nx.Graph, g_physical: nx.Graph,
                virtual_centrality_measure: int = 3, damping: float = 0.85,
                max_iter: int = 100, tol: float = 1e-6) -> dict:
    """
    Correct implementation of MCPR according to equation (7) of the paper:
    xi = αA ∑ Aij (xj/gj) + (1 - αA)yi

    Args:
        g_virtual: virtual layer network (for computing yi)
        g_physical: physical layer network (for PageRank)
        virtual_centrality_measure: centrality type for virtual layer (3=closeness recommended)
        damping: coefficient αA in the formula (default 0.85)
        max_iter: maximum number of iterations
        tol: convergence threshold

    Returns:
        dict: MCPR scores for each node
    """
    num_nodes = g_physical.number_of_nodes()
    nodes = list(g_physical.nodes())

    print(f"Starting MCPR computation for {num_nodes} nodes...")

    # Step 1: Compute yi from virtual layer centrality
    print("Computing virtual layer centrality...")
    centrality_vals = get_centrality(g_virtual, virtual_centrality_measure)

    # Normalize yi so its sum equals 1 (probability distribution)
    total_centrality = sum(centrality_vals.values())
    if total_centrality == 0:
        y = {node: 1.0 / num_nodes for node in nodes}
        print("Warning: total centrality is zero, using uniform distribution")
    else:
        y = {node: centrality_vals.get(node, 0) / total_centrality for node in nodes}

    # Step 2: Compute node degrees in physical layer
    degrees = {}
    for node in nodes:
        degree = g_physical.degree(node)
        degrees[node] = max(degree, 1)  # Prevent division by zero

    # Step 3: Initialize x (uniform distribution)
    x = {node: 1.0 / num_nodes for node in nodes}

    # Step 4: Power Iteration according to equation (7)
    print("Starting Power Iteration...")
    converged = False

    for iteration in range(max_iter):
        x_new = {}

        for node in nodes:
            # Compute ∑ Aij (xj/gj) for neighbors of node
            sum_term = 0.0
            neighbors = list(g_physical.neighbors(node))

            for neighbor in neighbors:
                # In undirected graph: Aij = 1 if (i,j) is an edge
                # sum_term += xj / gj
                sum_term += x[neighbor] / degrees[neighbor]

            # Apply equation (7): xi = αA * sum_term + (1 - αA) * yi
            x_new[node] = damping * sum_term + (1 - damping) * y[node]

        # Check convergence
        diff = sum(abs(x_new[node] - x[node]) for node in nodes)
        x = x_new

        if diff < tol:
            print(f"MCPR converged at iteration {iteration + 1}")
            converged = True
            break

        if iteration % 10 == 0:
            print(f"Iteration {iteration}: difference = {diff:.8f}")

    if not converged:
        print(f"Warning: MCPR did not converge in {max_iter} iterations")

    return x


def classic_immunization(graph: nx.Graph, measure_code: int, num_immunized: int) -> list:
    """
    Classic immunization based on single-layer centrality
    """
    centrality_scores = get_centrality(graph, measure_code)
    sorted_nodes = sorted(centrality_scores, key=centrality_scores.get, reverse=True)
    return sorted_nodes[:num_immunized]


def aware_immunization(g_virtual: nx.Graph, g_physical: nx.Graph,
                      num_immunized: int, virtual_centrality_measure: int = 3) -> list:
    """
    MCPR immunization using the correct implementation
    """
    mcp_ranking = mcpr_correct(g_virtual, g_physical, virtual_centrality_measure)
    sorted_nodes = sorted(mcp_ranking, key=mcp_ranking.get, reverse=True)
    return sorted_nodes[:num_immunized]


def multiplex_pagerank(g_layer1: nx.Graph, g_layer2: nx.Graph,
                      num_immunized: int, q: float = 0.5, n: float = 0.5) -> list:
    """
    Multiplex PageRank (MPR) implementation according to the paper
    """
    num_nodes = g_layer1.number_of_nodes()

    # Compute PageRank of layer 1
    pr_layer1 = nx.pagerank(g_layer1, alpha=0.85)
    x_layer1 = np.array([pr_layer1.get(i, 0) for i in range(num_nodes)])

    # Compute personalization for layer 2
    x_mean = np.mean(x_layer1)
    personalization = {}
    for i in range(num_nodes):
        if x_mean > 0:
            personalization[i] = (x_layer1[i] / x_mean) ** n
        else:
            personalization[i] = 1.0 / num_nodes

    # Normalize personalization
    total_pers = sum(personalization.values())
    if total_pers > 0:
        personalization = {k: v/total_pers for k, v in personalization.items()}

    # Compute PageRank of layer 2 with personalization
    pr_layer2 = nx.pagerank(g_layer2, alpha=0.85, personalization=personalization)

    # Combine results
    combined_ranking = {}
    for i in range(num_nodes):
        combined_ranking[i] = q * x_layer1[i] + (1-q) * pr_layer2.get(i, 0)

    sorted_nodes = sorted(combined_ranking, key=combined_ranking.get, reverse=True)
    return sorted_nodes[:num_immunized]


# =============================================================================
# Section 2: SIR-UA Simulation Model
# =============================================================================

def simulate_sir_ua_single_run(args: tuple) -> int:
    """
    A single run of the SIR-UA model simulation
    """
    g_contact, g_com, initial_states, initial_awareness, beta, mu, phi, gamma, alpha, max_steps, num_seed_nodes, seed = args
    np.random.seed(seed)

    states = np.copy(initial_states)
    awareness = np.copy(initial_awareness)

    # Select initial infection nodes
    susceptible_nodes = [node for node, state in enumerate(states) if state != 3]
    if len(susceptible_nodes) < num_seed_nodes:
        return 0

    infected_nodes = np.random.choice(susceptible_nodes, num_seed_nodes, replace=False)
    states[infected_nodes] = 2
    awareness[infected_nodes] = -1  # Initial awareness of infected nodes
    epidemic_size = len(infected_nodes)

    # Step-by-step simulation
    for step in range(max_steps):
        currently_infected = np.where(states == 2)[0]
        if len(currently_infected) == 0:
            break

        # Phase 1: Recovery
        recovery_probs = np.random.rand(len(currently_infected))
        recovered_nodes = currently_infected[recovery_probs <= mu]
        states[recovered_nodes] = 3
        awareness[recovered_nodes] = 0

        currently_infected = np.where(states == 2)[0]
        if len(currently_infected) == 0:
            break

        # Phase 2: Infection spread
        nodes_to_infect = []
        for i_node in currently_infected:
            neighbors = list(g_contact.neighbors(i_node))
            susceptible_neighbors = [n for n in neighbors if states[n] in [0, 1]]

            for s_node in susceptible_neighbors:
                if states[s_node] == 0:  # Unaware susceptible (US)
                    infection_rate = beta  # βU
                else:  # Aware susceptible (AS_i)
                    # βA = (1 - φ^i) * βU according to equation (2)
                    awareness_level = max(0, awareness[s_node])
                    infection_rate = (1 - phi**awareness_level) * beta

                if np.random.rand() < infection_rate:
                    nodes_to_infect.append(s_node)

        if nodes_to_infect:
            unique_new_infections = np.unique(nodes_to_infect)
            states[unique_new_infections] = 2
            awareness[unique_new_infections] = -1
            epidemic_size += len(unique_new_infections)

        # Phase 3: Information spread
        aware_nodes = np.where(awareness >= 0)[0]
        nodes_to_update_awareness = {}

        for a_node in aware_nodes:
            neighbors = list(g_com.neighbors(a_node))
            if not neighbors:
                continue

            for neighbor in neighbors:
                if np.random.rand() < alpha:
                    if states[neighbor] == 0:  # US -> AS_1
                        states[neighbor] = 1
                        nodes_to_update_awareness[neighbor] = 1
                    elif states[neighbor] == 1 and awareness[neighbor] > awareness[a_node] + 1:
                        # Update with better awareness
                        nodes_to_update_awareness[neighbor] = min(
                            nodes_to_update_awareness.get(neighbor, float('inf')),
                            awareness[a_node] + 1
                        )

        for node, new_awareness in nodes_to_update_awareness.items():
            awareness[node] = new_awareness

        # Phase 4: Information quality decay
        aware_nodes_for_fading = np.where(awareness > 0)[0]
        fading_probs = np.random.rand(len(aware_nodes_for_fading))
        nodes_to_fade = aware_nodes_for_fading[fading_probs < gamma]
        awareness[nodes_to_fade] += 1

    return epidemic_size


def run_sir_ua_parallel(g_contact, g_com, initial_states, initial_awareness,
                       beta, mu, phi, gamma, alpha, max_steps, num_seed_nodes, iterations):
    """
    Parallel execution of SIR-UA simulation
    """
    args_list = []
    for i in range(iterations):
        args_list.append((
            g_contact, g_com, initial_states, initial_awareness,
            beta, mu, phi, gamma, alpha, max_steps, num_seed_nodes,
            int(time.time() * 1000 + i) % (2**32)  # unique seed
        ))

    with Pool(processes=min(cpu_count(), iterations)) as pool:
        results = pool.map(simulate_sir_ua_single_run, args_list)

    # Filter valid results
    num_nodes = g_contact.number_of_nodes()
    valid_results = [res for res in results if res > 0.001 * num_nodes]

    return np.mean(valid_results) if valid_results else 0.0


# =============================================================================
# Section 3: Loading and Creating Networks - Modified for CSV Dataset
# =============================================================================

def load_networks_from_csv(physical_path: str, virtual_path: str):
    """
    Load networks from CSV files of the high school dataset
    Modified to work correctly with the provided datasets
    """
    print(f"Loading physical network from: {physical_path}")

    try:
        # Check if files exist
        if not os.path.exists(physical_path):
            raise FileNotFoundError(f"File {physical_path} not found")
        if not os.path.exists(virtual_path):
            raise FileNotFoundError(f"File {virtual_path} not found")

        # Physical network (contacts) - standard CSV format with comma
        print("Reading physical network file...")
        df_physical = pd.read_csv(physical_path, header=None, names=['source', 'target'])
        print(f"Number of physical edges: {len(df_physical)}")
        print(f"Physical data sample: {df_physical.head()}")

        # Create physical network
        g_physical = nx.from_pandas_edgelist(df_physical, source='source', target='target')
        g_physical.remove_edges_from(nx.selfloop_edges(g_physical))  # Remove self-loops

        print(f"Loading virtual network from: {virtual_path}")

        # Virtual network (friendships) - standard CSV format with comma
        print("Reading virtual network file...")
        df_virtual = pd.read_csv(virtual_path, header=None, names=['source', 'target'])
        print(f"Number of virtual edges: {len(df_virtual)}")
        print(f"Virtual data sample: {df_virtual.head()}")

        # Create virtual network
        g_virtual = nx.from_pandas_edgelist(df_virtual, source='source', target='target')
        g_virtual.remove_edges_from(nx.selfloop_edges(g_virtual))  # Remove self-loops

        # Unify node sets
        all_nodes_set = set(g_physical.nodes()) | set(g_virtual.nodes())
        all_nodes = sorted(list(all_nodes_set))

        print(f"Total unique nodes: {len(all_nodes)}")
        print(f"Node range: {min(all_nodes)} to {max(all_nodes)}")

        # Create mapping for nodes (starting from 0)
        mapping = {old_node: new_node for new_node, old_node in enumerate(all_nodes)}

        # Apply mapping
        g_physical = nx.relabel_nodes(g_physical, mapping)
        g_virtual = nx.relabel_nodes(g_virtual, mapping)

        # Ensure all nodes exist in both networks
        num_nodes = len(all_nodes)
        g_physical.add_nodes_from(range(num_nodes))
        g_virtual.add_nodes_from(range(num_nodes))

        print(f"\n=== Network Summary ===")
        print(f"Number of nodes: {num_nodes}")
        print(f"Physical network: {g_physical.number_of_edges()} edges")
        print(f"Virtual network: {g_virtual.number_of_edges()} edges")
        print(f"Physical density: {nx.density(g_physical):.4f}")
        print(f"Virtual density: {nx.density(g_virtual):.4f}")

        # Check connectivity
        physical_components = nx.number_connected_components(g_physical)
        virtual_components = nx.number_connected_components(g_virtual)
        print(f"Physical connected components: {physical_components}")
        print(f"Virtual connected components: {virtual_components}")

        if physical_components > 1:
            largest_cc = max(nx.connected_components(g_physical), key=len)
            print(f"Largest physical connected component size: {len(largest_cc)}")

        if virtual_components > 1:
            largest_cc = max(nx.connected_components(g_virtual), key=len)
            print(f"Largest virtual connected component size: {len(largest_cc)}")

        return g_physical, g_virtual

    except Exception as e:
        print(f"Error loading networks: {e}")
        print("Error details:")
        import traceback
        traceback.print_exc()
        raise


def create_synthetic_networks(num_nodes: int = 2000):
    """
    Create synthetic networks according to paper specifications (Table 4)
    """
    print(f"Creating synthetic networks with {num_nodes} nodes...")

    # Physical layer: scale-free network with distribution P(k) ~ k^-2.5
    g_physical = nx.scale_free_graph(num_nodes, alpha=0.41, beta=0.54, gamma=0.05, seed=42)
    g_physical = g_physical.to_undirected()
    g_physical.remove_edges_from(nx.selfloop_edges(g_physical))

    # Virtual layer: add 800 random edges
    g_virtual = g_physical.copy()

    nodes = list(g_virtual.nodes())
    edges_added = 0
    max_attempts = 5000
    attempts = 0

    np.random.seed(42)
    while edges_added < 800 and attempts < max_attempts:
        u, v = np.random.choice(nodes, 2, replace=False)
        if not g_virtual.has_edge(u, v):
            g_virtual.add_edge(u, v)
            edges_added += 1
        attempts += 1

    print(f"Physical network: {g_physical.number_of_nodes()} nodes, {g_physical.number_of_edges()} edges")
    print(f"Virtual network: {g_virtual.number_of_nodes()} nodes, {g_virtual.number_of_edges()} edges")

    return g_physical, g_virtual


# =============================================================================
# Section 4: Analysis and Display of Results
# =============================================================================

def plot_results(results_df, disease_type):
    """
    Plot comparison results of immunization methods
    """
    plt.style.use('default')
    fig, ax = plt.subplots(figsize=(14, 10))

    strategies = [col for col in results_df.columns if col != "Immunization %"]
    colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
    markers = ['o', 's', '^', 'v', 'D', 'P', '*', 'X']

    for i, strategy in enumerate(strategies):
        ax.plot(results_df["Immunization %"], results_df[strategy],
                marker=markers[i % len(markers)], linestyle='-', linewidth=2.5,
                color=colors[i % len(colors)], label=strategy, markersize=8)

    ax.set_title(f'Comparison of Immunization Strategies - {disease_type.capitalize()}',
                fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Immunization Percentage (%)', fontsize=14)
    ax.set_ylabel('Average Epidemic Size', fontsize=14)

    ax.set_xticks(results_df["Immunization %"])
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.grid(True, alpha=0.3, linestyle='--')

    # Legend settings
    ax.legend(fontsize=11, frameon=True, shadow=True,
             bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()

    output_filename = f'immunization_results_{disease_type}_mcpr_corrected.png'
    plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nPlot saved to '{output_filename}'.")

    plt.show()


def analyze_mcpr_performance(g_virtual, g_physical, top_k=10):
    """
    Analyze MCPR performance and compare with other methods
    """
    print("=== MCPR Performance Analysis ===\n")

    # Compute various scores
    print("Computing centrality scores...")

    # MCPR with closeness
    mcpr_closeness = mcpr_correct(g_virtual, g_physical, virtual_centrality_measure=3)

    # Classical PageRank
    classical_pr = nx.pagerank(g_physical, alpha=0.85)

    # MCPR with different centrality measures
    centrality_names = {1: "Degree", 2: "Betweenness", 3: "Closeness",
                       4: "Eigenvector", 5: "PageRank"}

    mcpr_results = {}
    for code, name in centrality_names.items():
        try:
            mcpr_results[name] = mcpr_correct(g_virtual, g_physical,
                                            virtual_centrality_measure=code)
        except Exception as e:
            print(f"Error computing MCPR with {name}: {e}")
            mcpr_results[name] = None

    # Display results
    print(f"\nTop {top_k} nodes according to different methods:\n")

    # Classical PageRank
    top_classical = sorted(classical_pr.items(), key=lambda x: x[1], reverse=True)[:top_k]
    print("Classical PageRank:")
    for i, (node, score) in enumerate(top_classical, 1):
        print(f"  {i:2d}. Node {node}: {score:.6f}")

    print()

    # MCPR variants
    for name, scores in mcpr_results.items():
        if scores is not None:
            top_nodes = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
            print(f"MCPR ({name}):")
            for i, (node, score) in enumerate(top_nodes, 1):
                print(f"  {i:2d}. Node {node}: {score:.6f}")
            print()

    # Overlap analysis
    print("Overlap analysis in top nodes:")
    classical_set = set([node for node, _ in top_classical])

    for name, scores in mcpr_results.items():
        if scores is not None:
            top_nodes = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
            mcpr_set = set([node for node, _ in top_nodes])
            overlap = len(classical_set & mcpr_set)
            diff_percent = ((top_k - overlap) / top_k) * 100
            print(f"  MCPR ({name}) vs Classical: {overlap}/{top_k} overlap ({diff_percent:.1f}% difference)")

    return mcpr_results, classical_pr


# =============================================================================
# Section 5: Comprehensive Experiment
# =============================================================================

def run_complete_experiment():
    """
    Run the complete MCPR experiment
    """
    print("="*60)
    print("Starting comprehensive MCPR experiment")
    print("="*60)

    # Settings
    SIMULATION_ITERATIONS = 20
    MAX_STEPS = 100
    DISEASE_TYPE = 'smallpox'

    # Disease parameters (according to Table 5 of the paper)
    DISEASE_PARAMS = {
        'measles': {'beta_u': 0.9, 'mu': 1/5},
        'smallpox': {'beta_u': 0.2, 'mu': 1/10}
    }
    BETA, MU = DISEASE_PARAMS[DISEASE_TYPE]['beta_u'], DISEASE_PARAMS[DISEASE_TYPE]['mu']

    # SIR-UA model parameters
    PHI, ALPHA, GAMMA = 0.8, 0.3, 0.1
    IMMUNIZATION_PERCENTAGES = np.arange(1, 11, 1)

    # Load networks
    PHYSICAL_NETWORK_FILE = "physical_graph_edgelist.csv"
    VIRTUAL_NETWORK_FILE = "facebook_graph_edgelist (1).csv"

    try:
        g_physical, g_virtual = load_networks_from_csv(PHYSICAL_NETWORK_FILE, VIRTUAL_NETWORK_FILE)
        print("Real networks loaded.")
    except Exception as e:
        print(f"Error loading dataset files: {e}")
        print("Creating synthetic networks...")
        g_physical, g_virtual = create_synthetic_networks(2000)

    NUM_NODES = g_physical.number_of_nodes()
    print(f"Network size: {NUM_NODES} nodes")
    print(f"Physical edges: {g_physical.number_of_edges()}")
    print(f"Virtual edges: {g_virtual.number_of_edges()}")

    # Initial MCPR analysis
    print("\nStep 1: MCPR Performance Analysis")
    mcpr_analysis, classical_analysis = analyze_mcpr_performance(g_virtual, g_physical)

    # Simulation experiments
    print("\nStep 2: Immunization Simulations")

    results = {
        "Immunization %": [],
        "Classical (PageRank)": [],
        "MCPR (Degree)": [],
        "MCPR (Betweenness)": [],
        "MCPR (Closeness)": [],
        "MCPR (Eigenvector)": [],
        "MCPR (PageRank)": [],
        "MPR (Virtual->Physical)": [],
        "MPR (Physical->Virtual)": []
    }

    for perc in IMMUNIZATION_PERCENTAGES:
        num_immunized = int(np.floor(perc / 100 * NUM_NODES))
        num_seed_nodes = int(np.floor(0.2 * NUM_NODES))  # According to paper

        print(f"\n--- Immunization experiment {perc}% ({num_immunized} nodes) ---")
        results["Immunization %"].append(perc)

        # Strategy 1: Classical PageRank
        start_time = time.time()
        nodes_to_immunize_classic = classic_immunization(g_physical, 5, num_immunized)
        initial_states_classic = np.zeros(NUM_NODES, dtype=int)
        initial_awareness_classic = -np.ones(NUM_NODES, dtype=int)
        initial_states_classic[nodes_to_immunize_classic] = 3
        size_classic = run_sir_ua_parallel(
            g_physical, g_virtual, initial_states_classic, initial_awareness_classic,
            BETA, MU, PHI, ALPHA, GAMMA, MAX_STEPS, num_seed_nodes, SIMULATION_ITERATIONS
        )
        results["Classical (PageRank)"].append(size_classic)
        print(f"Classical PageRank: {size_classic:.2f} (time: {time.time() - start_time:.2f}s)")

        # Strategy 2-6: MCPR with different centrality measures
        centrality_names = ["Degree", "Betweenness", "Closeness", "Eigenvector", "PageRank"]
        for i, centrality_name in enumerate(centrality_names, 1):
            start_time = time.time()
            try:
                nodes_to_immunize_mcpr = aware_immunization(g_virtual, g_physical, num_immunized, i)
                initial_states_mcpr = np.zeros(NUM_NODES, dtype=int)
                initial_awareness_mcpr = -np.ones(NUM_NODES, dtype=int)
                initial_states_mcpr[nodes_to_immunize_mcpr] = 3
                size_mcpr = run_sir_ua_parallel(
                    g_physical, g_virtual, initial_states_mcpr, initial_awareness_mcpr,
                    BETA, MU, PHI, ALPHA, GAMMA, MAX_STEPS, num_seed_nodes, SIMULATION_ITERATIONS
                )
                results[f"MCPR ({centrality_name})"].append(size_mcpr)
                print(f"MCPR ({centrality_name}): {size_mcpr:.2f} (time: {time.time() - start_time:.2f}s)")
            except Exception as e:
                print(f"Error in MCPR ({centrality_name}): {e}")
                results[f"MCPR ({centrality_name})"].append(float('inf'))

        # Strategy 7-8: MPR variants
        start_time = time.time()
        nodes_to_immunize_mpr1 = multiplex_pagerank(g_virtual, g_physical, num_immunized)
        initial_states_mpr1 = np.zeros(NUM_NODES, dtype=int)
        initial_awareness_mpr1 = -np.ones(NUM_NODES, dtype=int)
        initial_states_mpr1[nodes_to_immunize_mpr1] = 3
        size_mpr1 = run_sir_ua_parallel(
            g_physical, g_virtual, initial_states_mpr1, initial_awareness_mpr1,
            BETA, MU, PHI, ALPHA, GAMMA, MAX_STEPS, num_seed_nodes, SIMULATION_ITERATIONS
        )
        results["MPR (Virtual->Physical)"].append(size_mpr1)
        print(f"MPR (Virtual->Physical): {size_mpr1:.2f} (time: {time.time() - start_time:.2f}s)")

        start_time = time.time()
        nodes_to_immunize_mpr2 = multiplex_pagerank(g_physical, g_virtual, num_immunized)
        initial_states_mpr2 = np.zeros(NUM_NODES, dtype=int)
        initial_awareness_mpr2 = -np.ones(NUM_NODES, dtype=int)
        initial_states_mpr2[nodes_to_immunize_mpr2] = 3
        size_mpr2 = run_sir_ua_parallel(
            g_physical, g_virtual, initial_states_mpr2, initial_awareness_mpr2,
            BETA, MU, PHI, ALPHA, GAMMA, MAX_STEPS, num_seed_nodes, SIMULATION_ITERATIONS
        )
        results["MPR (Physical->Virtual)"].append(size_mpr2)
        print(f"MPR (Physical->Virtual): {size_mpr2:.2f} (time: {time.time() - start_time:.2f}s)")

    # Display final results
    print("\n" + "="*60)
    print("Final Experiment Results")
    print("="*60)

    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False, float_format='%.2f'))

    # Plot results
    plot_results(results_df, DISEASE_TYPE)

    # Statistical analysis
    print("\n" + "="*60)
    print("Performance Analysis")
    print("="*60)

    final_results = results_df[results_df["Immunization %"] == 10]

    best_strategy = ""
    best_score = float('inf')

    for strategy in results_df.columns[1:]:
        if strategy in final_results.columns:
            avg_score = results_df[strategy].mean()
            final_score = final_results[strategy].iloc[0]
            print(f"{strategy:<25}: average = {avg_score:6.2f}, final (10%) = {final_score:6.2f}")

            if final_score < best_score:
                best_score = final_score
                best_strategy = strategy

    print(f"\n🏆 Best strategy: {best_strategy}")
    print(f"   Epidemic size at 10% immunization: {best_score:.2f}")

    # Compute improvement relative to Classical PageRank
    if "Classical (PageRank)" in final_results.columns:
        classical_score = final_results["Classical (PageRank)"].iloc[0]
        if classical_score > 0:
            improvement = ((classical_score - best_score) / classical_score) * 100
            print(f"📈 Improvement over Classical PageRank: {improvement:.1f}%")

    return results_df, mcpr_analysis


# =============================================================================
# Section 6: Test and Verification Functions
# =============================================================================

def test_mcpr_correctness():
    """
    Test the correctness of MCPR implementation with a small network
    """
    print("=== Testing Correctness of MCPR Implementation ===")

    # Create small test network
    g_physical = nx.Graph()
    g_physical.add_edges_from([(0,1), (1,2), (2,3), (0,3), (1,3)])

    g_virtual = nx.Graph()
    g_virtual.add_edges_from([(0,2), (1,3), (2,4), (0,4)])

    # Ensure same nodes
    all_nodes = set(g_physical.nodes()) | set(g_virtual.nodes())
    g_physical.add_nodes_from(all_nodes)
    g_virtual.add_nodes_from(all_nodes)

    print(f"Test network: {len(all_nodes)} nodes")
    print(f"Physical edges: {list(g_physical.edges())}")
    print(f"Virtual edges: {list(g_virtual.edges())}")

    # Test MCPR
    mcpr_scores = mcpr_correct(g_virtual, g_physical, virtual_centrality_measure=3)
    classical_scores = nx.pagerank(g_physical, alpha=0.85)

    print("\nResults:")
    print("Node\tMCPR\t\tPageRank")
    print("-" * 30)
    for node in sorted(all_nodes):
        print(f"{node}\t{mcpr_scores[node]:.6f}\t{classical_scores[node]:.6f}")

    # Check constraints
    mcpr_sum = sum(mcpr_scores.values())
    pr_sum = sum(classical_scores.values())
    print(f"\nMCPR sum: {mcpr_sum:.6f}")
    print(f"PageRank sum: {pr_sum:.6f}")

    return mcpr_scores, classical_scores


def compare_centrality_measures():
    """
    Compare different centrality measures for the virtual layer
    """
    print("=== Comparing Centrality Measures ===")

    # Create sample network
    g = nx.karate_club_graph()

    measures = {
        1: "Degree",
        2: "Betweenness",
        3: "Closeness",
        4: "Eigenvector",
        5: "PageRank"
    }

    results = {}
    for code, name in measures.items():
        try:
            centrality = get_centrality(g, code)
            top_nodes = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:5]
            results[name] = top_nodes

            print(f"\nTop 5 - {name}:")
            for i, (node, score) in enumerate(top_nodes, 1):
                print(f"  {i}. Node {node}: {score:.4f}")

        except Exception as e:
            print(f"Error in {name}: {e}")

    return results


def test_dataset_loading():
    """
    Specific test for loading the provided datasets
    """
    print("=== Testing Dataset Loading ===")

    physical_file = "physical_graph_edgelist.csv"
    virtual_file = "facebook_graph_edgelist (1).csv"

    print(f"Checking file existence...")
    print(f"Physical file: {os.path.exists(physical_file)}")
    print(f"Virtual file: {os.path.exists(virtual_file)}")

    if os.path.exists(physical_file) and os.path.exists(virtual_file):
        try:
            g_physical, g_virtual = load_networks_from_csv(physical_file, virtual_file)

            # Quick analysis
            print(f"\n=== Quick Analysis ===")
            print(f"Nodes: {g_physical.number_of_nodes()}")
            print(f"Physical edges: {g_physical.number_of_edges()}")
            print(f"Virtual edges: {g_virtual.number_of_edges()}")

            # Test MCPR on a small sample
            if g_physical.number_of_nodes() > 100:
                print("\nTesting MCPR on first 100 nodes...")
                subgraph_phys = g_physical.subgraph(range(100)).copy()
                subgraph_virt = g_virtual.subgraph(range(100)).copy()

                mcpr_test = mcpr_correct(subgraph_virt, subgraph_phys, virtual_centrality_measure=3)
                print(f"MCPR computed for {len(mcpr_test)} nodes")

                # Display top 5
                top_mcpr = sorted(mcpr_test.items(), key=lambda x: x[1], reverse=True)[:5]
                print("Top 5 nodes by MCPR:")
                for i, (node, score) in enumerate(top_mcpr, 1):
                    print(f"  {i}. Node {node}: {score:.6f}")

            return g_physical, g_virtual

        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()
            return None, None
    else:
        print("Files not found!")
        return None, None


# =============================================================================
# Section 7: Main Function
# =============================================================================

def main():
    """
    Main function to run the complete project
    """
    print("Starting MCPR Project - Implementation According to Paper")
    print("Modified version to work with CSV datasets")
    print("=" * 60)

    # Select execution type
    print("Select an option:")
    print("1. Test dataset loading")
    print("2. Quick correctness test of implementation")
    print("3. Compare centrality measures")
    print("4. Run complete experiment (time-consuming)")

    try:
        choice = input("Your choice (1-4): ").strip()

        if choice == "1":
            test_dataset_loading()

        elif choice == "2":
            test_mcpr_correctness()

        elif choice == "3":
            compare_centrality_measures()

        elif choice == "4":
            results_df, mcpr_analysis = run_complete_experiment()

            # Save results
            results_df.to_csv("mcpr_experiment_results.csv", index=False)
            print("\nResults saved to 'mcpr_experiment_results.csv'.")

        else:
            print("Invalid choice. Running dataset loading test...")
            test_dataset_loading()

    except KeyboardInterrupt:
        print("\nExperiment stopped by user.")
    except Exception as e:
        print(f"Unexpected error: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()